<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 6 — Scenario Tuning and Calibration
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_06.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 · 4 · 5 · 6 · 7 · **6 (this notebook)**

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | Implementing Rule NRB-VEL-002 (velocity) · two-rule alert overlap (mirrors Section 7.8 of the text) | — |
| **2. Exercise 7.1 Extension** | Rule responsiveness curves · cross-rule overlap analysis · combined alert prioritisation | Exercise 7.1 |
| **3. Reflection cells** | Structured answer prompts | Exercise 7.1 Parts A–C |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist. All account IDs, names, and transactions are generated from a fixed random seed for educational purposes only.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: Adding Rule NRB-VEL-002 (Velocity)

> *This section mirrors Section 7.8 of the textbook exactly. Run the cells and compare the output to the printed figures.*

### The two-rule system so far

| Rule ID | Chapter introduced | Detection logic |
|---------|-------------------|-----------------|
| NRB-STRUCT-001 | 4 | Rolling 30-day cash deposits > USD 7,500, ≥ 3 transactions |
| NRB-VEL-002 | 6 (this chapter) | ≥ 5 transactions within any 14-day window, with consecutive gaps ≤ 3 days |

Velocity rules detect accounts that transact at unusual frequency in short bursts. A mule running money through an account typically makes many rapid-fire deposits or withdrawals, not the irregular spending of an ordinary retail customer.

### Why both rules together?

Any account that triggers **both** rules is exhibiting two independent red flags simultaneously: structuring behaviour AND unusually rapid transaction cadence. The probability that both patterns arise by coincidence in a genuine retail customer is low. Accounts triggering both rules should receive higher investigation priority.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_txn = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
MULE_IDS = [f'ACC{i:04d}' for i in range(1, 7)]

def apply_rule_1(df_txn, threshold=7500, min_txns=3):
    cash = df_txn[
        (df_txn['txn_type'] == 'CASH_IN') &
        (df_txn['amount'] < 10_000)
    ].copy().sort_values(['account_id', 'txn_date'])
    results = []
    for acct, grp in cash.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rs = grp['amount'].rolling('30D').sum()
        rc = grp['amount'].rolling('30D').count()
        if rs.max() > threshold and rc.max() >= min_txns:
            results.append({'account_id': acct})
    return pd.DataFrame(results)

def apply_rule_2(df_txn, min_txns=5, window_days=14, max_gap_days=3):
    """Rule NRB-VEL-002: rapid-fire transactions within a short window."""
    df = df_txn.copy().sort_values(['account_id', 'txn_date'])
    results = []
    for acct, grp in df.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rolling_cnt = grp['amount'].rolling(f'{window_days}D').count()
        peak_cnt = rolling_cnt.max()
        if peak_cnt >= min_txns:
            gaps = grp.index.to_series().diff().dt.days.dropna()
            short_gaps = (gaps <= max_gap_days).sum()
            if short_gaps >= (min_txns - 1):
                results.append({
                    'account_id':    acct,
                    'peak_txn_count': int(peak_cnt),
                    'short_gap_count': int(short_gaps),
                })
    return pd.DataFrame(results)

alerts_r1 = apply_rule_1(df_txn)
alerts_r2 = apply_rule_2(df_txn)

print(f"Rule 1 (Structuring) alerts:  {len(alerts_r1)} accounts")
print(f"Rule 2 (Velocity) alerts:     {len(alerts_r2)} accounts")
print()
print("Rule 2 — top 10 accounts by peak transaction count:")
print(alerts_r2.sort_values('peak_txn_count', ascending=False).head(10).to_string(index=False))

In [ ]:
# Cross-rule overlap analysis
set_r1 = set(alerts_r1['account_id'])
set_r2 = set(alerts_r2['account_id'])
both   = set_r1 & set_r2
r1_only = set_r1 - set_r2
r2_only = set_r2 - set_r1

print(f"Rule 1 only:  {len(r1_only)} accounts")
print(f"Both rules:   {len(both)} accounts")
print(f"Rule 2 only:  {len(r2_only)} accounts")
print()
print("Accounts triggering BOTH rules:")
print(sorted(both)[:20])
print()
mule_in_both = [a for a in both if a in MULE_IDS]
print(f"Known mule accounts triggering both rules: {len(mule_in_both)} / 6")
print(mule_in_both)

In [ ]:
# Bar chart: alert overlap
fig, ax = plt.subplots(figsize=(6, 4))
categories = ['Rule 1 only\n(Structuring)', 'Both rules', 'Rule 2 only\n(Velocity)']
counts     = [len(r1_only), len(both), len(r2_only)]
colours    = ['#4472C4', '#E74C3C', '#2ECC71']
bars = ax.bar(categories, counts, color=colours, width=0.5)
ax.bar_label(bars, padding=4, fontsize=12, fontweight='bold')
ax.set_title('Alert Overlap: NRB-STRUCT-001 vs NRB-VEL-002', fontsize=11, fontweight='bold', pad=10)
ax.set_ylabel('Number of Accounts', fontsize=10)
ax.set_ylim(0, max(counts) * 1.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**What you're seeing:** Accounts triggering both rules (red bar) are a subset of all alerted accounts and represent the highest-priority investigation queue. Accounts triggering only Rule 1 or only Rule 2 still warrant review, but the dual-trigger accounts have two independent red flags — statistically less likely to be false positives.

Notice how many mule accounts fall in the "both rules" category. In Chapter 8, this dual-rule flag will be one of the input features to the Isolation Forest model.

---
## Section 2 — Exercise 7.1 Extension: Rule Responsiveness Analysis

> *The main-text exercise asks you to run both rules at default settings. This extension asks you to tune each rule's key parameters and measure how the alert population changes.*

In [ ]:
# Rule 2 responsiveness: vary the minimum transaction count threshold
min_txns_range = [3, 4, 5, 6, 7, 8]
r2_results = []

for m in min_txns_range:
    a2 = apply_rule_2(df_txn, min_txns=m)
    mule_hit = a2['account_id'].isin(MULE_IDS).sum() if len(a2) > 0 else 0
    r2_results.append({
        'min_txns':       m,
        'total_alerts':   len(a2),
        'mule_recall':    mule_hit,
        'false_positives': len(a2) - mule_hit,
    })

r2_sens = pd.DataFrame(r2_results)
print("Rule 2 Responsiveness — Varying Minimum Transaction Count (14-day window):")
print(r2_sens.to_string(index=False))

**✏️ YOUR OBSERVATION**

Look at the responsiveness table for Rule 2. As `min_txns` increases from 3 to 8:
- At what value does the rule first miss one or more mule accounts?
- What is the ratio of false positives to mule accounts at `min_txns=5` (the default)?
- For a bank with 15 investigators, which setting would you recommend?

*Write your answers in Section 3, Question 2.*

In [ ]:
# Combined alert priority matrix: all accounts, coded by rule overlap
all_accts = sorted(set_r1 | set_r2)
priority = []
for acct in all_accts:
    r1 = acct in set_r1
    r2 = acct in set_r2
    is_mule = acct in MULE_IDS
    priority.append({
        'account_id': acct,
        'rule1':      int(r1),
        'rule2':      int(r2),
        'both_rules': int(r1 and r2),
        'known_mule': int(is_mule),
        'priority':   'HIGH' if (r1 and r2) else 'MEDIUM',
    })

priority_df = pd.DataFrame(priority)
print("Alert Priority Summary:")
print(priority_df.groupby(['priority','known_mule']).size().unstack(fill_value=0))

---
## Section 3 — Reflection: Exercise 7.1 Answer Cells

> *Double-click any cell to edit it.*

#### Question 1 — Rule 2 Behaviour

*(Edit this cell to write your answer)*

**How many accounts triggered Rule NRB-VEL-002 at the default settings (min_txns=5, window_days=14)?**  
  
**How many mule accounts are captured by Rule 2? How does this compare to Rule 1?**  
  
**In plain English, describe the transaction pattern that Rule 2 is designed to detect.**  


#### Question 2 — Tuning Rule 2

*(Edit this cell to write your answer)*

**At which `min_txns` setting does Rule 2 first miss a mule account?**  
  
**What is the false positive rate at the default setting (min_txns=5)?**  
  
**If you had to choose between min_txns=4 and min_txns=6 for a bank with limited investigation capacity, which would you choose and why?**  


#### Question 3 — Multi-Rule Priority

*(Edit this cell to write your answer)*

**How many accounts triggered both rules at the default settings?**  
  
**Are all six mule accounts in the "both rules" group, or do some trigger only one rule? What does this suggest about the mule behaviour?**  
  
**How would you communicate the difference between HIGH and MEDIUM priority alerts to a Level 1 investigator?**  


---
## What's Next

In Chapter 8, you will add a third rule — **NRB-GEO-003** — which targets accounts transacting with high-risk country counterparties. You will then build the full three-rule coverage matrix and map each rule to the FFIEC red-flag typologies it is designed to detect.

Open Chapter 7: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_07.ipynb)